<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINT-Colab/blob/main/notebooks/08_geoclip_geolocalizacao_auditada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 08 · Geolocalização com GeoCLIP, auditada em quilômetros

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINT-Colab)

O **GeoCLIP** (NeurIPS 2023) devolve as 5 coordenadas mais prováveis de uma foto, com probabilidade. O OSINTIA mede se essa probabilidade é honesta **em distância**: quantas vezes a previsão cai a menos de 25 km (cidade), 200 km (região) e 750 km (país). A verdade vem do **GPS no EXIF das suas próprias fotos originais**, então você audita o modelo no seu tipo de imagem antes de usá-lo num caso.

| | |
|---|---|
| **Hardware** | GPU T4 gratuita (Ambiente de execução → Alterar tipo → T4 GPU) |
| **Tempo estimado** | 10 minutos (funciona em CPU, mais devagar) |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

In [ ]:
!pip install -q geoclip

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINT-Colab.git /content/OSINT-Colab 2>/dev/null || git -C /content/OSINT-Colab pull -q
import sys
sys.path.insert(0, "/content/OSINT-Colab")
from osintia import auditoria, calibracao, coleta, coordenacao, custodia, dados, esquemas, extracao, geo, roteamento
print("OSINTIA pronto")

## Parte A · A auditoria, sem GPU
Previsões **simuladas** de um geolocalizador superconfiante sobre 240 fotos fictícias em capitais brasileiras. Ele diz 50% a 80% de certeza até em fotos de natureza e interiores.

In [ ]:
sim = dados.carregar_jsonl("/content/OSINT-Colab/dados/demo/geo_previsoes.jsonl")
prev = [tuple(x["topk"][0]) for x in sim]
verd = [tuple(x["verdade"]) for x in sim]
conf = [x["probs"][0] for x in sim]

print("acurácia por distância:", geo.acuracia_por_distancia(prev, verd))
cal = geo.calibracao_distancia(conf, prev, verd, raio_km=25)
print(f"\nECE no raio de cidade (25 km): {cal['ece']}")
for linha in cal["tabela"]:
    print(linha)

Por tipo de cena: onde o modelo merece confiança?

In [ ]:
from collections import defaultdict
por_cena = defaultdict(list)
for x, p, v in zip(sim, prev, verd):
    por_cena[x["cena"]].append(geo.haversine_km(p, v) <= 25)
for cena, acertos in sorted(por_cena.items(), key=lambda kv: -sum(kv[1]) / len(kv[1])):
    print(f"{cena:<14} acerto em 25 km: {sum(acertos) / len(acertos):.0%}  (n={len(acertos)})")

**Sinal independente da probabilidade:** se as 5 previsões estão espalhadas, o modelo não sabe, mesmo que o top-1 diga 80%.

In [ ]:
estaveis = [geo.concordancia_topk([tuple(t) for t in x["topk"]], raio_km=200)["estavel"] for x in sim]
acerto = [geo.haversine_km(p, v) <= 25 for p, v in zip(prev, verd)]
for flag in (True, False):
    sel = [a for a, e in zip(acerto, estaveis) if e == flag]
    print(f"top-5 {'estável ' if flag else 'disperso'}: {len(sel):>3} fotos, acerto em 25 km = {sum(sel) / len(sel):.0%}")

## Parte B · GeoCLIP de verdade
Envie **fotos originais suas, com GPS** (do celular, sem passar por redes sociais, que apagam o EXIF). Fotos sem GPS também funcionam, mas entram só como previsão, sem auditoria.

In [ ]:
import torch
from geoclip import GeoCLIP
modelo = GeoCLIP()
if torch.cuda.is_available():
    modelo = modelo.to("cuda")

In [ ]:
from google.colab import files
from PIL import Image
enviados = list(files.upload())
print(len(enviados), "arquivos")

In [ ]:
def gps_da_foto(caminho):
    exif = Image.open(caminho).getexif()
    g = exif.get_ifd(0x8825) if exif else {}
    if 2 in g and 4 in g:
        return (geo.gps_exif_para_graus(g[2], g.get(1, "N")), geo.gps_exif_para_graus(g[4], g.get(3, "E")))
    return None

resultados = []
for arq in enviados:
    gps, prob = modelo.predict(arq, top_k=5)
    topk = [tuple(map(float, p)) for p in gps.tolist()]
    resultados.append({"arquivo": arq, "topk": topk, "probs": [float(x) for x in prob.tolist()],
                       "verdade": gps_da_foto(arq)})
    r = resultados[-1]
    extra = f" | erro: {geo.haversine_km(topk[0], r['verdade']):.1f} km" if r["verdade"] else " | sem GPS no EXIF"
    print(f"{arq}: top-1 {topk[0][0]:.4f}, {topk[0][1]:.4f} (p={r['probs'][0]:.3f}){extra}")

In [ ]:
com_gps = [r for r in resultados if r["verdade"]]
if len(com_gps) >= 5:
    p = [r["topk"][0] for r in com_gps]; v = [r["verdade"] for r in com_gps]; c = [r["probs"][0] for r in com_gps]
    print("acurácia nas suas fotos:", geo.acuracia_por_distancia(p, v))
    print("calibração (750 km):", geo.calibracao_distancia(c, p, v, raio_km=750))
else:
    print(f"{len(com_gps)} foto(s) com GPS: envie pelo menos 5 para uma auditoria com algum significado (30+ é o ideal).")

## Parte C · Mapa e custódia
O KML abre no Google Earth: pontos vermelhos são previsões, verdes são o GPS real.

In [ ]:
import json, os
os.makedirs("/content/saidas", exist_ok=True)
pontos = []
for r in resultados:
    for k, (lat, lon) in enumerate(r["topk"]):
        pontos.append({"nome": f"{r['arquivo']} #{k + 1}", "lat": lat, "lon": lon, "descricao": f"p={r['probs'][k]:.3f}"})
    if r["verdade"]:
        pontos.append({"nome": f"{r['arquivo']} GPS real", "lat": r["verdade"][0], "lon": r["verdade"][1]})
geo.kml_pontos(pontos, "/content/saidas/geoclip.kml", nome="OSINTIA GeoCLIP")
with open("/content/saidas/geoclip.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=1)
for r in resultados:
    custodia.registrar("/content/saidas/manifesto.jsonl", "coleta_imagem", r["arquivo"])
custodia.registrar("/content/saidas/manifesto.jsonl", "geolocalizacao_geoclip", "/content/saidas/geoclip.json")
print("KML e JSON em /content/saidas/")

## (Opcional) StreetCLIP: país ou cidade entre candidatos
Zero-shot entre opções que **você** define. Atenção à licença **CC BY-NC 4.0** (não comercial) e ao aviso do autor: geolocalizar imagens privadas de pessoas está fora do uso pretendido.

In [ ]:
USAR_STREETCLIP = False
CANDIDATOS = ["Belo Horizonte", "São Paulo", "Rio de Janeiro", "Recife", "Salvador", "Porto Alegre", "Curitiba", "Manaus"]
if USAR_STREETCLIP:
    from transformers import pipeline
    clf = pipeline("zero-shot-image-classification", model="geolocal/StreetCLIP")
    for arq in enviados:
        top = clf(Image.open(arq), candidate_labels=CANDIDATOS)[:3]
        print(arq, [(t["label"], round(t["score"], 3)) for t in top])

### Leitura
- A probabilidade do GeoCLIP é **relativa** a uma galeria de coordenadas; ela não vem calibrada. Por isso a Parte A.
- Use as previsões para **direcionar a busca** (imagens de satélite, Street View, grade KML da Bellingcat), nunca como resposta.
- Combine com o notebook 05: o Qwen3.5 explica as pistas em texto; o GeoCLIP aponta coordenadas.

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- `docs/COLABS-RELACIONADOS.md` lista outros notebooks de OSINT (Bellingcat e comunidade) que combinam com este.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*